# 03_Gold_Analytics

Run on Databricks serverless with the uploaded Olist files. Original team work: review, execute, validate and document results.

In [ ]:
CATALOG = "workspace"
SCHEMA = "ecommerce_bigdata"
VOLUME = "raw_data"
RAW_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print('Raw data path:', RAW_PATH)

In [ ]:
from pyspark.sql import functions as F
p=f"{CATALOG}.{SCHEMA}"
def s(name):return spark.table(f"{p}.silver_{name}")
def gold(df,name):
    df.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable(f"{p}.gold_{name}")
    print(name,df.count())
# Grain: one row per order item. Revenue = item price (freight excluded).
# Exclude canceled/unavailable orders and do not join payments directly (multi-payment duplication).
o=s('orders').filter(~F.col('order_status').isin('canceled','unavailable'))
i=s('order_items');c=s('customers');pr=s('products');rv=s('order_reviews')
fact=(i.join(o.select('order_id','customer_id','order_status','purchase_at','delivered_at','estimated_at'),'order_id','inner')
    .join(c.select('customer_id','customer_unique_id','customer_state','customer_city'),'customer_id','left')
    .join(pr.select('product_id','product_category_name_english'),'product_id','left')
    .join(rv,'order_id','left')
    .withColumn('order_month',F.date_format('purchase_at','yyyy-MM'))
    .withColumn('delivery_days',F.datediff('delivered_at','purchase_at'))
    .withColumn('is_late',F.when(F.col('delivered_at').isNotNull() & F.col('estimated_at').isNotNull(),(F.col('delivered_at')>F.col('estimated_at')).cast('int')))
    .withColumn('category',F.coalesce('product_category_name_english',F.lit('unknown'))))
gold(fact,'order_items_fact')
gold(fact.groupBy('order_month').agg(F.round(F.sum('price'),2).alias('revenue_brl'),F.countDistinct('order_id').alias('orders'),F.countDistinct('customer_unique_id').alias('customers'),F.round(F.avg('review_score'),2).alias('avg_review_score')).orderBy('order_month'),'monthly_sales')
gold(fact.groupBy('category').agg(F.round(F.sum('price'),2).alias('revenue_brl'),F.countDistinct('order_id').alias('orders'),F.sum(F.lit(1)).alias('units')).orderBy(F.desc('revenue_brl')),'category_sales')
gold(fact.groupBy('customer_state').agg(F.round(F.sum('price'),2).alias('revenue_brl'),F.countDistinct('order_id').alias('orders')).orderBy(F.desc('revenue_brl')),'state_sales')
# Delivery metrics at order grain, not item grain.
delivery=o.withColumn('delivery_days',F.datediff('delivered_at','purchase_at')).withColumn('late',F.when(F.col('delivered_at').isNotNull() & F.col('estimated_at').isNotNull(),(F.col('delivered_at')>F.col('estimated_at')).cast('int')))
gold(delivery.groupBy('order_status').agg(F.count('*').alias('orders'),F.round(F.avg('delivery_days'),2).alias('avg_delivery_days'),F.round(F.avg('late')*100,2).alias('late_pct')),'delivery_summary')
# Payment totals are separate at order grain.
gold(s('order_payments').groupBy('payment_type').agg(F.round(F.sum('payment_value'),2).alias('payment_value_brl'),F.countDistinct('order_id').alias('orders')),'payment_summary')